# LangChain Agent Class Demo: Search + Weather + RAG

This Colab project demonstrates a **tool-using LangChain agent** that can decide between:

1. **External Search Tool** — current web information using SerpAPI  
2. **Weather Tool** — live weather using OpenWeatherMap API  
3. **RAG Tool** — answers from your uploaded PDF using FAISS vector search

The idea is simple:

> A normal chatbot only answers from model knowledge.  
> An agent can choose tools: search the web, call APIs, or retrieve from your private document.

---

## Class Demo Scenario

Ask the agent questions like:

- `What is the latest news about ISRO?`
- `What is the weather in Mumbai?`
- `According to the uploaded PDF, what are the candidate's main skills?`
- `Compare the weather in Bengaluru with current news about traffic there.`

The agent decides which tool to call.


In [1]:
# ============================================================
# 1. Install required packages
# ============================================================

!pip install -q   langchain   langchain-community   langchain-openrouter   langchain-huggingface   sentence-transformers   faiss-cpu   pypdf   google-search-results   requests


## 2. Add API keys safely

You need three keys:

- `OPENROUTER_API_KEY` — for the LLM
- `SERPAPI_API_KEY` — for Google search through SerpAPI
- `OPENWEATHER_API_KEY` — for weather data

In class, explain that keys should not be hard-coded in notebooks. Use `getpass` instead.


In [2]:
# ============================================================
# 2. API keys
# ============================================================

import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OPENROUTER_API_KEY: ")
os.environ["SERPAPI_API_KEY"] = getpass.getpass("Enter SERPAPI_API_KEY: ")
os.environ["OPENWEATHER_API_KEY"] = getpass.getpass("Enter OPENWEATHER_API_KEY: ")

print("✅ API keys loaded into environment variables")


✅ API keys loaded into environment variables


## 3. Import libraries

This uses the newer LangChain `create_agent` interface plus standard `@tool` functions.


In [3]:
# ============================================================
# 3. Imports
# ============================================================

import os
import requests

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openrouter import ChatOpenRouter

from langchain_community.utilities import SerpAPIWrapper
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("✅ Imports complete")


d:\GitHub\Backend-Development-AI\venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


✅ Imports complete


## 4. Upload a PDF for RAG

For demo, upload any PDF such as:

- Resume
- HR policy
- Product document
- Company FAQ
- Course notes

The RAG tool will answer only from this uploaded document.


In [5]:
# ============================================================
# 4. Enter PDF Path
# ============================================================

import os

PDF_PATH = input("Enter the full path of the PDF: ").strip().strip('"')

if not os.path.isfile(PDF_PATH):
    raise FileNotFoundError(f"File not found: {PDF_PATH}")

print(f"✅ Selected PDF: {PDF_PATH}")

✅ Selected PDF: C:\Users\Mohankumar MC\Desktop\Mohankumar.pdf


## 5. Build FAISS Vector Store

This is the RAG part, similar to your reference notebook:

1. Load PDF pages  
2. Split pages into chunks  
3. Convert chunks to embeddings  
4. Store embeddings in FAISS  
5. Search relevant chunks during question answering


In [6]:
# ============================================================
# 5. Build the RAG vector store
# ============================================================

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total pages loaded: {len(documents)}")
print(f"Total chunks created: {len(chunks)}")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("✅ FAISS vector store created")


Total pages loaded: 2
Total chunks created: 13


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ FAISS vector store created


## 6. Create Tools

The agent gets three tools.

### Tool 1: Search
Use this when the question needs **latest/current information**.

### Tool 2: Weather
Use this when the question asks about **weather in a city**.

### Tool 3: RAG
Use this when the question asks about the **uploaded PDF**.


In [7]:
# ============================================================
# 6A. External Search Tool using SerpAPI
# ============================================================

@tool
def search_tool(query: str) -> str:
    """
    Search Google for current general information using SerpAPI.
    Do NOT use this tool for weather questions.
    For weather questions, use weather_tool only.
    """
    print("🔎 SERPAPI SEARCH TOOL CALLED")
    return SerpAPIWrapper(serpapi_api_key=os.environ["SERPAPI_API_KEY"]).run(query)


In [8]:
@tool
def weather_tool(location: str) -> str:
    """Use this tool ONLY for current weather questions. Input should be a city name."""
    print("🌦️ WEATHER TOOL CALLED")
    print("Location received:", location)

    api_key = os.environ.get("OPENWEATHER_API_KEY")

    if not api_key:
        return (
            "WEATHER_TOOL_ERROR: OPENWEATHER_API_KEY is missing. "
            "Do not call any other tool. Tell the user the weather API key is missing."
        )

    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": location,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params, timeout=10)

    print("Status code:", response.status_code)
    print("Raw response:", response.text)

    if response.status_code != 200:
        return (
            f"WEATHER_TOOL_ERROR: OpenWeather API failed with status code "
            f"{response.status_code}. Response: {response.text}. "
            "Do not call search_tool. Tell the user the weather API failed."
        )

    data = response.json()

    city = data["name"]
    country = data["sys"]["country"]
    condition = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    feels_like = data["main"]["feels_like"]
    humidity = data["main"]["humidity"]
    wind_speed = data["wind"]["speed"]

    return (
        f"FINAL_WEATHER_RESULT: Current weather in {city}, {country}: "
        f"{condition}, temperature {temp}°C, feels like {feels_like}°C, "
        f"humidity {humidity}%, wind speed {wind_speed} m/s. "
        "Use this result directly. Do not call search_tool."
    )

In [9]:
# ============================================================
# 6C. RAG Tool using FAISS Retriever
# ============================================================

@tool
def rag_tool(question: str) -> str:
    """Search the uploaded PDF and return relevant context. Use this when the user asks about the uploaded document, resume, policy, notes, or internal knowledge base."""
    print("📚 RAG TOOL CALLED")

    docs = retriever.invoke(question)

    if not docs:
        return "No relevant information found in the uploaded PDF."

    formatted_chunks = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page", "unknown")
        source = doc.metadata.get("source", PDF_PATH)
        formatted_chunks.append(
            f"Chunk {i} | Source: {source} | Page: {page}\n{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


## 7. Initialize the LLM

This version uses **OpenRouter** through the dedicated LangChain integration.

You can change the model if needed. For class demos, choose a reliable low-cost model available in your OpenRouter account.


In [10]:
# ============================================================
# 7. LLM using OpenRouter
# ============================================================

llm = ChatOpenRouter(
    model="openai/gpt-4o-mini",
    temperature=0.2,
    max_tokens = 150,
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("✅ LLM initialized")


✅ LLM initialized


## 8. Create the LangChain Agent

The system prompt tells the agent when to use which tool.

Important teaching point:

> The agent does not blindly call all tools.  
> It decides the right tool based on the user question.


In [11]:
# ============================================================
# 8. Create agent
# ============================================================

tools = [search_tool, weather_tool, rag_tool]

system_prompt = """
You are a helpful AI class-demo assistant.

You have access to three tools:

1. search_tool:
   Use for latest news, current facts, recent information, or external web knowledge.

2. weather_tool:
   Use for current weather, temperature, humidity, wind, or rain questions.

3. rag_tool:
   Use for questions about the uploaded PDF or private document.


Tool selection rules:
1. For weather questions, use weather_tool only.
2. Do not use search_tool for weather questions.
3. If weather_tool returns FINAL_WEATHER_RESULT, immediately answer using that result.
4. If weather_tool returns WEATHER_TOOL_ERROR, explain the error to the user. Do not call another tool.
5. For questions about uploaded PDFs/documents, use rag_tool.
6. For general current information, use search_tool.
7. Do not call more than one tool unless the user question clearly requires multiple tools.

"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

print("✅ Agent created")


✅ Agent created


## 9. Helper Function to Chat with Agent

This makes the demo easier to run.


In [12]:
# ============================================================
# 9. Ask helper
# ============================================================

def ask_agent(question: str):
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    final_message = result["messages"][-1]
    print("\n==============================")
    print("USER QUESTION:")
    print(question)
    print("\nAGENT ANSWER:")
    print(final_message.content)
    print("==============================\n")


## 10. Demo Queries

Run these one by one during class and show which tool gets called.


In [13]:
# Demo 1: RAG question from uploaded PDF
ask_agent("According to the uploaded PDF, summarize the key skills or main points.")


📚 RAG TOOL CALLED

USER QUESTION:
According to the uploaded PDF, summarize the key skills or main points.

AGENT ANSWER:
The uploaded PDF highlights the following key skills and main points:

1. **Certifications**:
   - Currently enrolled in the Airtribe AI Software Engineer Program, focusing on backend engineering, including JavaScript fundamentals, REST APIs, asynchronous programming, microservices, JWT authentication, unit and integration testing, CI/CD with GitHub Actions, and emerging technologies in RAG systems and LLM integration.
   - Completed the Great Learning Executive Program in Artificial Intelligence and Machine Learning, covering machine learning fundamentals, data analytics, exploratory data analysis, data preprocessing, and decision-making with data.

2. **Professional Experience**:
   - Worked as a Systems Engineer at Infosys Limited from December 2020 to July 2022.
     - Redesigned a



In [14]:
# Demo 2: Weather question
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the current weather in Kolkata?"
            }
        ]
    },
    config={"recursion_limit": 4}
)

print(response["messages"][-1].content)


🌦️ WEATHER TOOL CALLED
Location received: Kolkata
Status code: 200
Raw response: {"coord":{"lon":88.3697,"lat":22.5697},"weather":[{"id":500,"main":"Rain","description":"light rain","icon":"10n"}],"base":"stations","main":{"temp":27.15,"feels_like":31.28,"temp_min":27.15,"temp_max":27.15,"pressure":1003,"humidity":91,"sea_level":1003,"grnd_level":1002},"visibility":3088,"wind":{"speed":1.52,"deg":175,"gust":2.48},"rain":{"1h":0.87},"clouds":{"all":100},"dt":1785949116,"sys":{"country":"IN","sunrise":1785886766,"sunset":1785933939},"timezone":19800,"id":1275004,"name":"Kolkata","cod":200}
The current weather in Kolkata is light rain, with a temperature of 27.15°C. It feels like 31.28°C, with a humidity of 91% and a wind speed of 1.52 m/s.


In [15]:
# Demo 3: External search question
ask_agent("What is the latest news about ISRO?")


🔎 SERPAPI SEARCH TOOL CALLED

USER QUESTION:
What is the latest news about ISRO?

AGENT ANSWER:
Here are some of the latest news highlights about ISRO:

1. **Bharatiya Antariksh Hackathon (BAH)**: ISRO has announced the launch of the Bharatiya Antariksh Hackathon, scheduled for July 06-10, 2026. They are also recruiting for various positions.

2. **Gaganyaan Mission**: ISRO successfully conducted the Second Integrated Air Drop Test (IADT-02) for the Gaganyaan mission, which is India's first human spaceflight program.

3. **Resignations at ISRO**: Recent reports indicate a concerning trend, with over 100 top scientists and engineers resigning from ISRO in the past few months.

4. **Chand



In [16]:
# Demo 4: Multi-tool question
ask_agent("Check the current weather in Bengaluru and also search for latest Bengaluru traffic news.")


🌦️ WEATHER TOOL CALLED
Location received: Bengaluru
🔎 SERPAPI SEARCH TOOL CALLED

USER QUESTION:
Check the current weather in Bengaluru and also search for latest Bengaluru traffic news.

AGENT ANSWER:
### Current Weather in Bengaluru
- **Condition:** Overcast clouds
- **Temperature:** 20.95°C (feels like 20.95°C)
- **Humidity:** 71%
- **Wind Speed:** 4.66 m/s

### Latest Bengaluru Traffic News
1. **Congestion Levels:** Bengaluru is experiencing significant congestion, with travel times and speeds based on extensive trip data indicating worsening conditions.
2. **Accidents and Breakdowns:** A vehicle breakdown occurred at Hosur Road, causing severe congestion near Swamy Vivekananda Road. An accident near Jakkur flyover has also led to slow traffic towards the airport.
3. **Traffic Management:** A delivery agent turned into a traffic cop amid a massive



## 11. Student Exercise

Ask students to add one more tool.

Ideas:

1. Calculator tool  
2. Currency converter tool  
3. Wikipedia tool  
4. SQL database tool  
5. Company FAQ tool  

Example starter:

```python
@tool
def calculator_tool(expression: str) -> str:
    """Evaluate a basic math expression."""
    return str(eval(expression))
```

For production, never use unrestricted `eval`. Use safe parsing instead.


## 12. Teaching Explanation

### What is happening internally?

1. User asks a question.
2. The agent reads the question.
3. The LLM decides whether it needs a tool.
4. If needed, it calls one or more tools.
5. Tool output comes back to the LLM.
6. The LLM writes the final answer.

### Difference from normal RAG

Your earlier notebook did this:

> User question → Vector search → LLM answer

This agent does this:

> User question → Decide tool → Search / Weather / RAG → LLM answer

So this is more flexible and closer to real business agents.
